# Credit Card Fraud Detection using Machine Learning

## 1. Problem Definition
Credit card fraud detection is a **binary classification** problem where each transaction must be labeled as either:
- **0 -> Non-Fraud / Legitimate transaction**
- **1 -> Fraudulent transaction**

This is an important real-world problem because missed frauds cause direct financial losses, customer distrust, chargebacks, and operational risk for banks and payment platforms.

In practice, fraud datasets are usually **highly imbalanced**, meaning fraudulent transactions are very rare compared with legitimate ones. Because of this, a model can achieve high accuracy while still missing most fraud cases. That is why this notebook focuses strongly on **recall, precision, F1-score, ROC-AUC, and confusion matrix**, not only accuracy.


## 2. Workflow Covered in This Notebook


1. **Data preprocessing**
   - missing-value check
   - feature scaling
   - encoding check
   - Principal Component Analysis (PCA)
   - class balancing with SMOTE
2. **Model building**
   - Logistic Regression
   - Random Forest
   - Support Vector Machine (SVM)
3. **Model evaluation**
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - ROC-AUC
   - Confusion Matrix
4. **Model optimization**
   - cross-validation
   - hyperparameter tuning with GridSearchCV
5. **Interpretation**
   - feature importance for Random Forest
6. **Conclusion**
   - best model selection and trade-off discussion


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
RANDOM_STATE = 42


## 3. Load Dataset
The notebook looks for `creditcard.csv` in common locations. If the file is not found, place the Kaggle dataset CSV in the same folder as this notebook or update the path list below.


In [ ]:
candidate_paths = [
    Path('creditcard.csv'),
    Path.cwd() / 'creditcard.csv',
    Path('/Users/ankitkumarsingh/Desktop/CREDIT-CARD-FRAUD-DETECTION-main/creditcard.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        'creditcard.csv was not found. Download the Kaggle credit card fraud dataset and place the CSV next to the notebook.'
    )

df = pd.read_csv(data_path)
print(f'Dataset loaded from: {data_path}')
print(f'Shape: {df.shape}')
df.head()


## 4. Initial Data Understanding
This section checks the dataset structure, missing values, and class imbalance.

### Why class imbalance matters
Fraud detection datasets contain very few fraud cases compared with normal transactions. If we train directly on the original data, many models become biased toward the majority class and may predict almost everything as legitimate.

### Why SMOTE is needed
**SMOTE (Synthetic Minority Over-sampling Technique)** creates synthetic minority-class examples by interpolating between existing fraud samples. This helps the model learn fraud patterns better without simply duplicating the same rare observations.

Important note: **SMOTE must be applied only on the training data**, never before the train-test split, otherwise it causes data leakage and makes evaluation overly optimistic.


In [ ]:
print('Data types:')
display(df.dtypes.to_frame('dtype').T)

missing_summary = df.isna().sum().sort_values(ascending=False)
print(f"Total missing values in dataset: {missing_summary.sum()}")
display(missing_summary[missing_summary > 0].to_frame('missing_count'))

class_distribution = df['Class'].value_counts().sort_index()
class_percent = df['Class'].value_counts(normalize=True).sort_index() * 100
summary_table = pd.DataFrame({
    'count': class_distribution.values,
    'percentage': class_percent.values,
}, index=['Non-Fraud (0)', 'Fraud (1)'])
display(summary_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='Class', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Original Class Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

class_percent.plot(kind='bar', ax=axes[1], color=['#4C72B0', '#DD8452'])
axes[1].set_title('Class Percentage Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## 5. Data Preprocessing

### Missing values
The missing-value check above tells us whether imputation is required. For this dataset, missing values are generally absent, but the check is still necessary in any real ML pipeline.

### Encoding
No categorical encoding is required here because the dataset contains only numeric features.

### Feature scaling
`StandardScaler` is used because models such as **Logistic Regression** and **SVM** are sensitive to feature scale.

### Principal Component Analysis (PCA)
Although the dataset already contains anonymized PCA-like variables (`V1` to `V28`), we still apply **PCA as a preprocessing step** to satisfy the dimensionality reduction requirement and to compress information before feeding it to scale-sensitive models.

### Train-test split
We split the data before any sampling so that the test set remains completely unseen.


In [ ]:
X = df.drop(columns='Class')
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print('Training class distribution:')
print(y_train.value_counts(normalize=True).rename('proportion'))


In [ ]:
# Demonstration of scaling + PCA on training data only
scaler_demo = StandardScaler()
X_train_scaled_demo = scaler_demo.fit_transform(X_train)
X_test_scaled_demo = scaler_demo.transform(X_test)

pca_demo = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca_demo = pca_demo.fit_transform(X_train_scaled_demo)
X_test_pca_demo = pca_demo.transform(X_test_scaled_demo)

print(f'Original feature count: {X_train.shape[1]}')
print(f'PCA components retained (95% variance): {pca_demo.n_components_}')
print(f'Cumulative explained variance: {pca_demo.explained_variance_ratio_.sum():.4f}')

plt.figure(figsize=(10, 4))
plt.plot(np.cumsum(pca_demo.explained_variance_ratio_), marker='o')
plt.axhline(0.95, color='red', linestyle='--', label='95% variance threshold')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()


## 6. Model Pipelines
We use three models:

- **Logistic Regression** as a simple and interpretable baseline
- **Random Forest** as the main tree-based model
- **Support Vector Machine (SVM)** as an additional strong classifier

For **Logistic Regression** and **SVM**, the pipeline includes:
`StandardScaler -> PCA -> SMOTE -> Model`

For **Random Forest**, the pipeline includes:
`SMOTE -> Model`

Random Forest does not require feature scaling and is kept on the original feature space so that we can later compute **feature importance** in terms of the original variables.


In [ ]:
pipelines = {
    'Logistic Regression': ImbPipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': ImbPipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', RandomForestClassifier(
            n_estimators=250,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight='balanced_subsample'
        ))
    ]),
    'Support Vector Machine': ImbPipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', LinearSVC(random_state=RANDOM_STATE, max_iter=5000))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
}


## 7. Cross-Validation
Cross-validation gives a more reliable estimate than a single split because the model is evaluated across multiple folds. In fraud detection, **recall** is especially important because failing to catch a fraudulent transaction is often costlier than a false alarm.


In [ ]:
cv_rows = []
for model_name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score='raise',
    )
    cv_rows.append({
        'Model': model_name,
        'CV Accuracy': scores['test_accuracy'].mean(),
        'CV Precision': scores['test_precision'].mean(),
        'CV Recall': scores['test_recall'].mean(),
        'CV F1': scores['test_f1'].mean(),
        'CV ROC-AUC': scores['test_roc_auc'].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values(by='CV Recall', ascending=False).reset_index(drop=True)
cv_results


## 8. Hyperparameter Tuning for Random Forest
To improve the main model, we tune Random Forest using **GridSearchCV** and optimize for **recall**, because identifying fraud is the top business priority in this project.


In [ ]:
rf_grid = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [None, 12, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 0.5],
}

rf_search = GridSearchCV(
    estimator=pipelines['Random Forest'],
    param_grid=rf_grid,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

rf_search.fit(X_train, y_train)
print('Best Parameters:', rf_search.best_params_)
print(f'Best Cross-Validated Recall: {rf_search.best_score_:.4f}')

best_rf_pipeline = rf_search.best_estimator_
pipelines['Random Forest (Tuned)'] = best_rf_pipeline


## 9. Final Evaluation on Test Data
The comparison below uses the untouched test set and includes all major metrics required for a strong fraud detection project.


In [ ]:
def get_model_scores(pipeline, X_test):
    if hasattr(pipeline, 'predict_proba'):
        return pipeline.predict_proba(X_test)[:, 1]
    if hasattr(pipeline, 'decision_function'):
        return pipeline.decision_function(X_test)
    raise AttributeError('Model does not support probability or decision scores.')

def evaluate_model(model_name, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_scores = get_model_scores(pipeline, X_test)

    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_scores),
    }
    return metrics, y_pred, y_scores

model_outputs = {}
comparison_rows = []
final_models = {
    'Logistic Regression': pipelines['Logistic Regression'],
    'Support Vector Machine': pipelines['Support Vector Machine'],
    'Random Forest': pipelines['Random Forest (Tuned)'],
}

for model_name, pipeline in final_models.items():
    metrics, y_pred, y_scores = evaluate_model(model_name, pipeline, X_train, X_test, y_train, y_test)
    comparison_rows.append(metrics)
    model_outputs[model_name] = {
        'pipeline': pipeline,
        'y_pred': y_pred,
        'y_scores': y_scores,
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'report': classification_report(y_test, y_pred, output_dict=True),
    }

comparison_df = pd.DataFrame(comparison_rows).sort_values(by=['Recall', 'ROC-AUC'], ascending=False).reset_index(drop=True)
comparison_df


## 10. Comparison Table
This table is useful for reports, viva discussion, and presentation slides because it clearly shows how the models differ across multiple metrics.


In [ ]:
styled_comparison = comparison_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-score': '{:.4f}',
    'ROC-AUC': '{:.4f}',
}).background_gradient(cmap='YlGn', subset=['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC'])
styled_comparison


## 11. Confusion Matrix Visualization
For fraud detection, the most important cell to watch is **False Negatives (actual fraud predicted as non-fraud)**. A lower false negative count means the system is missing fewer fraud cases.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (model_name, output) in zip(axes, model_outputs.items()):
    sns.heatmap(
        output['confusion_matrix'],
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False,
        ax=ax,
    )
    ax.set_title(f'{model_name} Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()


## 12. ROC Curve Visualization
ROC curves help compare the ranking quality of the models across different classification thresholds.


In [ ]:
plt.figure(figsize=(8, 6))
for model_name, output in model_outputs.items():
    fpr, tpr, _ = roc_curve(y_test, output['y_scores'])
    auc_score = roc_auc_score(y_test, output['y_scores'])
    plt.plot(fpr, tpr, linewidth=2, label=f'{model_name} (AUC = {auc_score:.4f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.tight_layout()
plt.show()


## 13. Feature Importance from Random Forest
Feature importance helps explain which variables contribute most to fraud detection. This adds interpretability and makes the project stronger during viva or presentation.


In [ ]:
rf_model = model_outputs['Random Forest']['pipeline'].named_steps['model']
feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_,
}).sort_values(by='Importance', ascending=False)

display(feature_importance_df.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_df.head(10),
    x='Importance',
    y='Feature',
    palette='viridis'
)
plt.title('Top 10 Important Features - Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 14. Interpretation and Conclusion
Use the output generated above to write the final conclusion in your report:

- **Best model:** choose the model with the strongest balance of recall, F1-score, and ROC-AUC.
- **Why recall matters most:** in fraud detection, missing fraud is more harmful than occasionally flagging a legitimate transaction.
- **Trade-off:** models with very high recall may increase false positives, so fraud systems usually balance customer experience with security.
- **Real-world applicability:** this workflow can support bank transaction monitoring, fraud alert systems, and risk-scoring pipelines.

### Suggested conclusion template
> Random Forest performed best overall because it achieved the strongest fraud detection performance on the test set, especially on recall and ROC-AUC. Logistic Regression provided a simple baseline, while SVM offered a competitive alternative after scaling, PCA, and SMOTE. Since fraud detection is an imbalanced classification problem, metrics such as recall, F1-score, ROC-AUC, and confusion matrix were more informative than accuracy alone.


## 15. Optional Extension
For extra presentation value, you can build a simple **Streamlit UI** where a user enters transaction details and the trained model predicts whether the transaction is fraudulent or legitimate.
